# Profiling Data: Indonesia Wikipedia Pages
- **Tools:** Polars (Lazy Evaluation) & DuckDB (SQL Analytics).
- **Tujuan:** Memvalidasi skema, menghitung total baris, dan mengecek anomali langsung dari disk tanpa membebani RAM.

In [4]:
import polars as pl
import duckdb
import os

DATA_PATH = '../data/raw/wikipedia_id-clean.csv'

file_size_mb = os.path.getsize(DATA_PATH) / (1024**2)
print(f"Ukuran file: {file_size_mb:.2f} MB")
assert file_size_mb >= 500, "Validasi Gagal: Dataset di bawah 500 MB"

Ukuran file: 1023.89 MB


In [7]:
print("--- POLARS: INSPEKSI SKEMA & SAMPEL ---")
# Menggunakan lazy scan agar tidak memuat 1 GB ke RAM
q = pl.scan_csv(DATA_PATH)

# Menggunakan collect_schema() sesuai instruksi dari peringatan Polars
print(q.collect_schema())

# Mengambil 5 baris pertama secara efisien
display(q.head(5).collect())

--- POLARS: INSPEKSI SKEMA & SAMPEL ---
Schema({'id': Int64, 'parent_id': Float64, 'timestamp': String, 'hash': String, 'title': String, 'text': String})


id,parent_id,timestamp,hash,title,text
i64,f64,str,str,str,str
1,1.9472643e7,"""2022-04-24T00:35:51Z""","""oo2rvi3f9p4u9czu3opfvbsl0jp9ao…","""Asam deoksiribonukleat""","""Asam deoksiribonukleat, lebih …"
3,1.9395623e7,"""2022-05-05T11:26:32Z""","""lq3uz0weougma29prpmkm1l5at48s4…","""Anwar Sadat""","""Muhammad Anwar el-Sadat; ) ada…"
4,1.4292725e7,"""2021-07-05T02:41:20Z""","""3h1z248a5ic1u3odrse27wg0o0sllj…","""Azhar Mansor""","""Datuk Azhar Mansor adalah oran…"
5,2.1368465e7,"""2022-09-26T04:25:33Z""","""ololfquhgvu4wzszfu1yaqo3amb8xn…","""Arkeologi""","""Arkeologi atau ilmu kepurbakal…"
6,2.1768805e7,"""2022-11-02T02:33:49Z""","""mkxdb4xfmgse21h6lvtoflnvwlepdb…","""Antropologi""","""Antropologi adalah ilmu tentan…"


In [6]:
print("--- DUCKDB: ANALITIK SQL ---")
# DuckDB dapat mengeksekusi SQL langsung ke file CSV di disk
query_profiling = f"""
SELECT 
    COUNT(*) as total_baris,
    COUNT(id) as total_id_valid,
    COUNT(*) - COUNT(title) as missing_titles
FROM '{DATA_PATH}'
"""

# Menjalankan kueri dan menampilkannya sebagai DataFrame Polars
hasil_kueri = duckdb.sql(query_profiling).pl()
display(hasil_kueri)

--- DUCKDB: ANALITIK SQL ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

total_baris,total_id_valid,missing_titles
i64,i64,i64
638798,638798,0
